In [1]:
#load the cleaned dataset
import pandas as pd

df = pd.read_csv("../processed/cleaned_trades.csv")

In [2]:
# basic sanity checks
df.shape
df.head()
df.isnull().sum()

account             0
coin                0
execution_price     0
size_tokens         0
size_usd            0
side                0
timestamp_ist       0
start_position      0
direction           0
closed_pnl          0
transaction_hash    0
order_id            0
crossed             0
fee                 0
trade_id            0
timestamp           0
time                0
Date                0
date                0
classification      0
dtype: int64

In [3]:
# data inspection
df.head(5)

,account,coin,execution_price,size_tokens,size_usd,side,timestamp_ist,start_position,direction,closed_pnl,transaction_hash,order_id,crossed,fee,trade_id,timestamp,time,Date,date,classification
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12,2024-10-27 03:33:20,2024-10-27,2024-10-27,Greed
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12,2024-10-27 03:33:20,2024-10-27,2024-10-27,Greed
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12,2024-10-27 03:33:20,2024-10-27,2024-10-27,Greed
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,02-12-2024 22:50,1146.558564,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,1.730000e+12,2024-10-27 03:33:20,2024-10-27,2024-10-27,Greed
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,02-12-2024 22:50,1289.488521,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,1.730000e+12,2024-10-27 03:33:20,2024-10-27,2024-10-27,Greed


In [7]:
# drop the duplicate
df = df.drop(columns=['Date'], errors='ignore')
df[['date', 'time']].head()

,date,time
0,2024-10-27,2024-10-27 03:33:20
1,2024-10-27,2024-10-27 03:33:20
2,2024-10-27,2024-10-27 03:33:20
3,2024-10-27,2024-10-27 03:33:20
4,2024-10-27,2024-10-27 03:33:20


In [17]:
# feature engineering
#1. Daily PnL: profit n loss per day
daily_pnl = df.groupby(['account', 'date'])['closed_pnl'].sum()

In [18]:
#win rate: % traders that are profitable
df['win'] = (df['closed_pnl'] > 0).astype(int)

win_rate = df.groupby(['account', 'date'])['win'].mean()

In [19]:
#profitable day flag: traders profit overall day
profitable_day = (daily_pnl > 0).astype(int)

In [20]:
#PnL Volatility (Risk proxy)
#trader’s returns are
# High volatility = risky / inconsistent
pnl_volatility = daily_pnl.groupby('account').std()

In [21]:
# behavioral metrics

In [22]:
# number of trades: how active a trader is
num_trades = df.groupby(['account', 'date'])['closed_pnl'].count()

In [24]:
# average trade size: capital per trade
avg_size = df.groupby(['account', 'date'])['size_usd'].mean()

In [30]:
df['side'].value_counts()
df['side'] = df['side'].str.lower()

In [31]:
# long Vs short ratio: more long= greed
long_short = df.groupby(['date', 'classification', 'side']).size().unstack(fill_value=0)

long_short['long_ratio'] = long_short['buy'] / (long_short['buy'] + long_short['sell'])

In [27]:
# frequency: high vs low traders
trade_counts = df.groupby('account')['closed_pnl'].count()

threshold = trade_counts.median()

df['frequency'] = df['account'].apply(
    lambda x: 'High' if trade_counts[x] > threshold else 'Low'
)

In [28]:
#trader quality: winner vs loser
total_pnl = df.groupby('account')['closed_pnl'].sum()

df['trader_quality'] = df['account'].apply(
    lambda x: 'Winner' if total_pnl[x] > 0 else 'Loser'
)

In [32]:
# risk proxy: largest size= highest risk
avg_size = df.groupby(['account', 'date'])['size_usd'].mean()

In [33]:
#final agg data
daily = df.groupby(['account', 'date', 'classification']).agg(
    daily_pnl=('closed_pnl', 'sum'),
    win_rate=('win', 'mean'),
    num_trades=('closed_pnl', 'count'),
    avg_size=('size_usd', 'mean')
).reset_index()

In [34]:
df.columns
df['side'].value_counts()

side
sell    95885
buy     88378
Name: count, dtype: int64

In [35]:
daily.to_csv("../processed/daily_metrics.csv", index=False)





In [36]:
num_trades = num_trades.reset_index()
num_trades.rename(columns={'closed_pnl': 'trades_per_day'}, inplace=True)

In [37]:
df = df.merge(num_trades, on=['account', 'date'], how='left')

In [38]:
df['activity_group'] = df['trades_per_day'].apply(
    lambda x: 'Low' if x <= 5 else ('Medium' if x <= 20 else 'High')
)

In [39]:
df[['account', 'date', 'trades_per_day', 'activity_group']].head()

,account,date,trades_per_day,activity_group
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,2024-10-27,190,High
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,2024-10-27,190,High
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,2024-10-27,190,High
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,2024-10-27,190,High
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,2024-10-27,190,High


In [42]:
df.to_csv("../processed/final_dataset.csv", index=False)

In [43]:
df.columns

Index(['account', 'coin', 'execution_price', 'size_tokens', 'size_usd', 'side',
       'timestamp_ist', 'start_position', 'direction', 'closed_pnl',
       'transaction_hash', 'order_id', 'crossed', 'fee', 'trade_id',
       'timestamp', 'time', 'date', 'classification', 'win', 'frequency',
       'trader_quality', 'trades_per_day', 'activity_group'],
      dtype='object')

In [53]:
trade_counts.describe()

count       32.000000
mean      5758.218750
std       7589.402695
min        332.000000
25%       1113.250000
50%       3152.500000
75%       7694.250000
max      36534.000000
Name: closed_pnl, dtype: float64

In [54]:
q1 = trade_counts.quantile(0.33)
q2 = trade_counts.quantile(0.66)

In [55]:
def activity_bucket(x):
    if x <= q1:
        return 'Low'
    elif x <= q2:
        return 'Medium'
    else:
        return 'High'

df['activity_group'] = df['account'].apply(lambda x: activity_bucket(trade_counts[x]))

In [56]:
df['activity_group'].value_counts()

activity_group
High      145706
Medium     30391
Low         8166
Name: count, dtype: int64

In [58]:
df.to_csv("../processed/final_dataset_v3.csv", index=False)

In [59]:
final = df.groupby(['activity_group', 'classification']).agg({
    'closed_pnl': 'mean'
}).reset_index()

In [60]:
final.to_csv("../processed/segmentation.csv", index=False)